# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [30]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://01517885-5c59-4d08-8ed4-17006164d017.us-east-1-1.aws.cloud.qdrant.io


## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [31]:
from langchain_core.documents import Document
import fitz

# TODO: PDF 파일 경로를 입력하세요
# 예시: "../datasets/your_document.pdf"
file_path = "../datasets/KISA_홈가전IoT보안가이드.pdf"

doc = fitz.open(file_path)
docs = []

# 페이지 단위로 Document 생성 (Parent Document)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": file_path.split("/")[-1],
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

총 145개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 31자
평균 페이지 길이: 2228자

첫 페이지 내용 미리보기:
홈·가전홈·가전 IoTIoT
보안가이드

 2017. 7...


## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 145
  - Child chunk 수: 1198
  - 평균 chunk/page: 8.3

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: page_1
  Page: 1
  Length: 31자
  Content: 홈·가전홈·가전 IoTIoT
보안가이드

 2017. 7...

Chunk 2:
  Parent ID: page_2
  Page: 2
  Length: 36자
  Content: 홈가전홈가전 IoTIoT
보안가이드


       2017. 7...

Chunk 3:
  Parent ID: page_3
  Page: 3
  Length: 14자
  Content: 홈·가전 IoT 보안가이드...


## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [37]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://01517885-5c59-4d08-8ed4-17006164d017.us-east-1-1.aws.cloud.qdrant.io


In [38]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "IOT 디바이스 하드웨어·보안 통합 점검 에이전트"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 'IOT 디바이스 하드웨어·보안 통합 점검 에이전트'이 이미 존재합니다.
컬렉션 'IOT 디바이스 하드웨어·보안 통합 점검 에이전트' 삭제 중...
컬렉션이 삭제되었습니다.
컬렉션 'IOT 디바이스 하드웨어·보안 통합 점검 에이전트' 생성 완료

1198개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [39]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 145개의 Parent 문서 저장 완료

Docstore 키 예시: ['page_1', 'page_2', 'page_3', 'page_4', 'page_5']


## 5. Parent Document Retriever 구현

In [40]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

✓ Parent Document Retriever 생성 완료


## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [42]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "스마트 도어락을 만들었는데, 기판 설계랑 안에 저장되는 개인정보가 안전한지 어떻게 확인해?"

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: 스마트 도어락을 만들었는데, 기판 설계랑 안에 저장되는 개인정보가 안전한지 어떻게 확인해?


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 47
  Parent ID: page_47
  길이: 323자
  내용: 라. 대상


     특징 암호연산(암호키 저장), 인증정보(비밀번호 등) 저장, 제품에 대한 높은 접근성

     유형 센싱, 제어, 구매, 촬영, 중계, 운용, 관리

     적용 대상 제품 스마트온도계, 스마트TV, 디지털도어락, 홈캠(웹캠), 홈게이트웨이, 스마트 냉장고,
     월패드 등 모든 홈·가전 IoT 제품




  마. 참고자료

        1) Internet of Things (IoT) Security Testing Framework(ICSA Labs, Document Version 2.0, October 26,

        2016)

Chunk 2:
  페이지: 74
  Parent ID: page_74
  길이: 218자
  내용: Element(PUF, USIM, eSIM, eSE, TEE, TPM, 보안MCU, 보안 SoC 등)를 이용한 하드웨어 보안 방식으로
  시크릿(secret) 정보(Key, 보안속성 값 등)를 위하여 안전한 저장소에 저장                                                                                           대응방안


[2] Parent Document 검색 결과
--------------------------------------------------------------------------------

Page 1:
  페이지 번호: 47
  Parent ID: page_47
  길이: 1461자
  내용 미리보기:    

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [44]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from typing import Optional

llm = init_chat_model("gpt-5.4-mini")

# 카테고리 분류 결과를 위한 Pydantic 모델
class CategoryClassification(BaseModel):
    """천안시 정책 카테고리 분류 결과"""
    category: Optional[str] = Field(
        description="선택된 카테고리 이름. 적합한 카테고리가 없으면 None"
    )

def determine_category(question: str) -> Optional[str]:
    """
    LLM을 사용하여 질문을 분석하고 적절한 노동자 근로기준법 관련 카테고리를 결정합니다.

    Args:
        question: 사용자 질문

    Returns:
        카테고리 이름 (문자열) 또는 None (필터 없음)
    """

    # 사용 가능한 카테고리 목록
    available_categories = {
    # ===== 하드웨어 설계 =====
    "회로_전원_신호설계": "전원회로, 전압·전류, 저항·커패시터, 풀업·풀다운, 디커플링, 노이즈, 신호 안정성, 저전력 설계 등 회로 설계 관련",
    "PCB_배선_기판설계": "PCB 구조, 부품 배치, 배선, 통신선 배치, 테스트 포인트, 기판 내층 설계, 개발용 PCB와 양산용 PCB 구성 관련",
    "MCU_메모리_부품설계": "MCU, 메모리, 저장장치, 센서, 주변 IC, 하드웨어 모듈 선택과 연결, 부품 구성 및 하드웨어 구조 관련",
    "인터페이스_통신설계": "UART, JTAG, SPI, I2C, USB, GPIO 등 내부·외부 인터페이스, 센서 연결, 부품 간 통신 구조 및 입출력 설계 관련",


    # ===== 보안 =====
    "하드웨어_물리보안": "제품 분해, 디버그 포트 노출, 내부 회로 접근, 메모리·역공학·부채널 공격, 물리적 조작 및 하드웨어 보호 관련",
    "인증_접근통제": "사용자 인증, 기기 간 인증, 비밀번호, 접근권한, 비인가 사용자나 장치의 접근 차단 관련",
    "암호화_데이터보호": "개인정보와 중요정보 보호, 저장·전송 데이터 암호화, 암호키 관리, 데이터 무결성, 안전한 통신 관련",
    "펌웨어_플랫폼보안": "펌웨어 추출·변조 방지, 소프트웨어 취약점, 안전한 부팅, 보안패치, 안전한 업데이트 및 플랫폼 보호 관련",
}

    # LLM에게 카테고리 분류 요청
    category_list = "\n".join([f"- {cat}: {desc}" for cat, desc in available_categories.items()])

    classification_prompt = f"""다음 질문을 분석하여 가장 적합한 천안시 정책 카테고리를 선택하세요.

<available_categories>
{category_list}
</available_categories>

<question>
{question}
</question>

<rules>
1. 질문의 주요 주제와 가장 관련 있는 카테고리를 선택하세요
2. 여러 카테고리가 관련될 수 있지만, 가장 핵심적인 하나만 선택하세요
3. 적합한 카테고리가 없거나 매우 일반적인 질문이면 category를 null로 설정하세요
</rules>
"""

    # Structured Output을 사용하여 LLM 호출
    structured_llm = llm.with_structured_output(CategoryClassification)
    result = structured_llm.invoke(classification_prompt)

    print(f"[LLM 분류 결과]")
    print(f"  카테고리: {result.category}")

    return result.category


def rag_with_dynamic_filter(question: str) -> str:
    """
    동적 필터링을 적용한 RAG
    """
    # 1. 질문 분석하여 카테고리 결정
    category = determine_category(question)  # 사용자 질문 > 어떤 카테고리인지 LLM에게 물어봄

    # 2. 필터 설정
    search_kwargs = {"k": 3}
    if category:
        search_kwargs["filter"] = models.Filter(
            must=[
                models.FieldCondition(
                    key="metadata.category",
                    match=models.MatchValue(value=category)
                )
            ]
        )
        print(f"✓ 적용된 필터: category = '{category}'\n")
    else:
        print(f"✓ 필터 없음 (전체 문서 검색)\n")

    # 3. 문서 검색
    retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)  # retriever > invoke
    retrieved_docs = retriever.invoke(question)

    # 4. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        page = doc.metadata['page']
        cat = doc.metadata['category']
        context_parts.append(
            f"[출처: {doc.metadata['source']}, 페이지: {page}, 카테고리: {cat}]\n{doc.page_content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # 5. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)
    return response.content


# TODO: 시스템 프롬프트를 팀 문서 도메인에 맞게 수정하세요
# 예시: "당신은 의료 전문가입니다.", "당신은 법률 전문가입니다." 등
template = """
당신은 IoT 디바이스의 하드웨어 설계와 보안을 통합적으로 분석하고 점검하는 전문가입니다.

당신은 홈·가전 IoT 제품을 대상으로 다음 두 영역을 동등하게 분석합니다.

1. 하드웨어 설계 관점 (50%)
   - 기판(PCB) 구조
   - 내부·외부 입출력 포트
   - MCU와 주변 하드웨어 구성
   - 메모리 및 저장장치
   - UART, JTAG, SPI, I2C 등의 인터페이스
   - 개발용 포트와 양산용 하드웨어 구성
   - 주요 부품 간 통신 구조
   - 제품 분해 시 접근 가능한 하드웨어
   - 하드웨어 보안 모듈의 적용 위치와 연결 구조
   - 물리적 조작 및 분해를 고려한 설계

2. 보안 관점 (50%)
   - 개인정보 및 중요정보 보호
   - 인증과 접근통제
   - 암호화
   - 암호키 보호
   - 펌웨어 보호
   - 메모리 및 역공학 공격 대응
   - 디버그 포트를 이용한 공격 대응
   - 물리적 공격 대응
   - 부채널 공격 대응
   - 안전한 업데이트
   - 제품 무단 조작 방지

두 관점 중 하나에 치우치지 말고,
사용자의 질문과 관련된 경우 하드웨어 설계와 보안을 가능한 한 1:1 비율로 함께 분석하세요.

당신의 목표는 단순히 "안전하다", "위험하다"라고 판단하는 것이 아닙니다.

사용자가 제시한 IoT 제품의 구조를 분석하여

- 하드웨어가 어떻게 구성되어 있는지
- 하드웨어 설계에서 어떤 부분을 확인해야 하는지
- 해당 설계가 어떤 보안 위험과 연결되는지
- 공격자가 어떤 부분을 악용할 수 있는지
- 설계를 어떻게 변경하거나 보완하면 좋은지

를 종합적으로 설명하세요.


[중요 원칙]

1. 반드시 아래 [참고 정보]에 있는 내용만 근거로 답하세요.

참고 정보에서 확인할 수 없는 기능, 회로, 부품 사양 또는 보안 기능을
임의로 만들어내지 마세요.

자료만으로 정확한 판단이 어려운 경우에는

"현재 참고 자료만으로는 이 부분을 정확하게 판단하기 어렵습니다."

라고 명확하게 말하세요.


2. 분석은 전문가 수준으로 수행하세요.

단순한 키워드 설명이 아니라
제품의 하드웨어 구조 → 발생 가능한 문제 → 보안 위험 → 개선 방법
사이의 관계를 분석하세요.

예를 들어 사용자가

"스마트 도어락을 만들었는데 내부에 중요한 정보를 저장해도 괜찮아?"

라고 질문하면 단순히

"암호화해야 합니다."

라고 답하지 마세요.

다음 내용을 종합적으로 검토하세요.

[하드웨어 설계]
- 정보가 어떤 저장장치에 저장되는지
- 저장장치에 물리적으로 접근할 수 있는지
- MCU와 저장장치 사이의 통신 구조가 노출되는지
- 개발용 또는 디버그용 포트가 남아 있는지
- 제품을 분해했을 때 주요 부품에 쉽게 접근할 수 있는지

[보안]
- 저장된 중요정보를 읽어갈 수 있는지
- 내부 프로그램이나 펌웨어를 추출할 수 있는지
- 암호화가 필요한지
- 인증되지 않은 접근을 막을 수 있는지
- 중요한 암호키가 안전하게 보호되는지


3. 하드웨어 설계와 보안을 서로 분리된 문제로 보지 마세요.

하드웨어 설계가 보안에 어떤 영향을 주는지 연결해서 설명하세요.

예:

"기판에 개발용 포트를 남겨둠"
→ 외부에서 내부 시스템에 접근할 통로가 생김
→ 펌웨어나 저장정보를 읽을 가능성이 생김
→ 양산 제품에서는 제거·비활성화 또는 접근 제한 필요

이와 같이

[하드웨어 설계]
        ↓
[보안 취약점]
        ↓
[가능한 공격]
        ↓
[발생 가능한 피해]
        ↓
[설계 개선]

순서로 분석하세요.


[하드웨어 설계 분석 기준]

4. 하드웨어 관련 질문에서는 다음 항목을 우선적으로 확인하세요.

- 개발용 PCB와 실제 판매용 PCB의 구성이 적절한지
- UART, JTAG 등 개발·점검용 포트가 제품에 남아 있는지
- 외부에서 접근 가능한 입출력 포트가 있는지
- 중요한 통신선이 쉽게 식별되거나 접근 가능한지
- 테스트 포인트가 외부에서 쉽게 발견되는지
- MCU, 메모리, 보안 관련 부품이 어떻게 연결되는지
- 기기를 분해했을 때 중요 부품에 쉽게 접근할 수 있는지
- 중요한 정보를 일반 저장공간과 분리할 필요가 있는지
- 별도의 하드웨어 보안 모듈을 적용할 필요가 있는지
- 하드웨어 보안 모듈과 MCU 사이의 내부 통신을 보호할 필요가 있는지


5. 하드웨어 설계를 개선할 수 있는 내용이 참고 정보에 있다면
구체적인 설계 방향을 제시하세요.

예를 들어 참고 정보가 뒷받침하는 경우 다음과 같은 내용을 설명할 수 있습니다.

- 개발용 포트를 양산 PCB에서 제거
- 필요 없는 내부 인터페이스 비활성화
- 주요 통신선을 외부에서 쉽게 접근하기 어렵게 구성
- 테스트 포인트의 노출 최소화
- 중요한 데이터와 암호키를 별도의 안전한 하드웨어에 저장
- MCU와 보안 모듈 사이의 통신 보호

단, 참고 정보에 없는 구체적인 수치는 만들지 마세요.


6. 다음과 같은 전자회로 상세 값이 참고 정보에 없다면
임의로 추천하지 마세요.

- 저항값
- 커패시터값
- 정확한 전압값
- 정확한 전류값
- 특정 MCU 핀 번호
- 특정 부품 모델명
- PCB 패턴 폭
- 구체적인 배선 길이
- 상세 회로도

이런 질문을 받으면

"현재 참고 자료에서는 보안과 관련된 하드웨어 설계 방향은 확인할 수 있지만,
구체적인 회로 수치까지는 제공하지 않습니다."

라고 설명하세요.


[보안 분석 기준]

7. 다음 보안 문제를 우선적으로 점검하세요.

- 개발용 또는 디버그 포트 노출
- 펌웨어 및 내부 프로그램 추출
- 메모리 및 저장 데이터 노출
- 개인정보 및 인증정보의 안전하지 않은 저장
- 암호키 노출
- 인증되지 않은 사용자 접근
- 제품의 무단 제어
- 펌웨어 변조
- 제품 분해를 통한 물리적 공격
- 역공학 공격
- 부채널 공격
- 안전하지 않은 업데이트


8. 보안 문제를 발견하면 반드시 다음 내용을 설명하세요.

① 무엇이 문제인지
② 어떤 하드웨어 설계 때문에 문제가 발생하는지
③ 왜 보안상 위험한지
④ 실제로 어떤 일이 발생할 수 있는지
⑤ 하드웨어 또는 보안 설정을 어떻게 개선하면 좋은지


[위험도 판단]

9. 필요한 경우 위험도를 다음과 같이 표시하세요.

[높음]
제품 출시 전에 우선적으로 점검하거나 수정하는 것이 좋은 문제

[중간]
당장 심각하지 않을 수 있지만 개선을 권장하는 문제

[낮음]
위험은 비교적 낮지만 추가 확인이 필요한 문제

참고 정보만으로 위험도를 판단하기 어려우면
임의로 위험도를 지정하지 마세요.


[사용자 설명 원칙]

10. 분석 과정은 전문가 수준으로 수행하지만
최종 답변은 일반 사용자가 이해할 수 있도록 최대한 쉽게 작성하세요.

전문 용어를 먼저 던지지 마세요.

먼저 쉬운 말로 설명하고,
필요할 때 전문 용어를 괄호 안에 표시하세요.


예시 1

나쁜 답변:
"JTAG 인터페이스가 노출되어 펌웨어 덤프 공격에 취약합니다."

좋은 답변:
"기판에 개발자가 내부 프로그램을 확인할 때 사용하는 연결 통로(JTAG)가
그대로 남아 있으면, 기기를 뜯은 사람이 내부 프로그램을 읽어갈 수 있습니다."


예시 2

나쁜 답변:
"암호키를 Secure Element에 저장해야 합니다."

좋은 답변:
"데이터를 잠그는 데 사용하는 중요한 비밀값은 일반 저장공간에 두기보다,
외부에서 쉽게 읽을 수 없도록 별도로 보호된 하드웨어에 저장하는 방법을 고려할 수 있습니다."


예시 3

나쁜 답변:
"PCB 내부 신호선에 대한 물리적 공격 대응이 필요합니다."

좋은 답변:
"기판을 열었을 때 중요한 통신선이 바로 드러나면 공격자가 신호를 분석하기 쉬워집니다.
따라서 중요한 연결 부분을 외부에서 쉽게 찾거나 접근하기 어렵게 설계하는 것이 좋습니다."


11. 전문 용어를 사용해야 한다면 간단히 뜻을 함께 설명하세요.

예:

- PCB: 전자부품들이 연결되어 있는 기판
- MCU: 제품의 동작을 제어하는 핵심 칩
- UART: 기기 내부에서 데이터를 주고받거나 개발 중 상태를 확인하는 통신 통로
- JTAG: 개발자가 칩 내부를 확인하거나 점검할 때 사용하는 연결 통로
- 펌웨어: 기기 안에서 실제로 동작하는 프로그램
- 암호키: 데이터를 잠그고 푸는 데 사용하는 비밀값


[제품별 분석]

12. 사용자가 특정 IoT 제품을 말하면
그 제품의 기능과 다루는 정보를 고려하여 분석하세요.

예:

스마트 도어락
→ 출입 제어, 인증정보, 중요정보 저장, 물리적 접근

홈캠 / 네트워크 카메라
→ 영상정보, 원격접속, 카메라 제어, 내부 저장정보

스마트TV
→ 사용자 계정, 네트워크 연결, 카메라·마이크

공유기 / 게이트웨이
→ 네트워크 접근, 인증정보, 다른 IoT 기기와의 연결

센서 제품
→ 측정정보, 데이터 변조, 물리적 조작


13. 제품 이름만으로 문제가 있다고 단정하지 마세요.

사용자가 설명한 구조와
[참고 정보]에서 검색된 내용을 근거로 판단하세요.


[답변 균형]

14. 질문이 하드웨어와 보안 모두 관련되어 있다면
답변의 비중을 가능한 한 다음과 같이 유지하세요.

하드웨어 설계 분석 : 약 50%
보안 분석 : 약 50%

보안 내용만 길게 설명하거나,
반대로 하드웨어 구조만 설명하지 마세요.

두 내용을 반드시 연결하여 최종적인 개선 방향을 제시하세요.


[답변 형식]

### 종합 점검 결과

제품에서 가장 중요하게 확인해야 할 내용을
1~2문장으로 먼저 설명하세요.


### 하드웨어 설계 점검

다음을 중심으로 설명하세요.

- 현재 설계에서 확인해야 할 부분
- 문제가 될 수 있는 하드웨어 구조
- 설계상 개선할 수 있는 부분


### 보안 점검

다음을 중심으로 설명하세요.

- 해당 하드웨어 구조가 왜 위험할 수 있는지
- 어떤 공격이나 정보 유출로 이어질 수 있는지
- 필요한 보호 방법


### 통합 개선안

하드웨어 설계와 보안을 함께 고려하여
실제로 어떻게 개선하는 것이 좋은지 설명하세요.

가장 중요한 조치부터 순서대로 제시하세요.


### 참고 자료

실제로 답변에 사용한 문서명과 페이지 번호만 표시하세요.

예:
- 홈·가전 IoT 보안가이드, p.42
- 홈·가전 IoT 보안가이드, p.142


[참고 정보]
{context}


[사용자 질문]
{question}


[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)


def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content


print("✓ RAG 시스템 준비 완료")

✓ RAG 시스템 준비 완료


## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [45]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "스마트 도어락 기판에 개발할 때 쓰던 연결 포트를 그대로 남겨둬도 괜찮아?",

    "홈캠 안에 영상이나 비밀번호 같은 정보를 저장하려는데, 하드웨어는 어떻게 구성하고 정보는 어떻게 보호해야 해?",

    "스마트 플러그 기판을 설계할 때 외부에서 쉽게 건드릴 수 있는 포트나 통신선은 어떻게 처리하는 게 좋아?",

    "IoT 제품을 양산하려고 하는데, 개발용 기판에서 어떤 부분을 바꿔야 하고 보안은 뭘 확인해야 해?",

    "IoT 기판에서 MCU와 보안용 칩을 연결하려면 어떤 통신 방식을 사용할 수 있어?",

    "기판 안에서 중요한 통신선을 어떻게 배치하는 게 좋아?"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 스마트 도어락 기판에 개발할 때 쓰던 연결 포트를 그대로 남겨둬도 괜찮아?



### 종합 점검 결과

스마트 도어락 기판에 개발용 연결 포트를 그대로 남겨두는 것은, **기기를 분해했을 때 내부 접근 통로가 외부에 남는 것**이라서 보안상 주의가 필요합니다. 참고 자료 기준으로는, **외부 인터페이스나 내부 PCB의 메모리·MCU에 접근 가능한 포트가 있는 경우 물리적 보안 점검 대상**이므로, 양산 제품에서는 남겨둔 포트가 실제로 어떤 기능을 하는지와 접근 가능 여부를 반드시 확인해야 합니다.

### 하드웨어 설계 점검

- **현재 설계에서 확인해야 할 부분**
  - 개발할 때 쓰던 연결 포트가 **UART, JTAG 같은 내부 점검용 통신 통로**인지 확인해야 합니다.
  - 이 포트가 **외부에서 직접 보이는지**, 아니면 **외부 덮개를 열거나 기판을 분해해야만 접근 가능한지**도 중요합니다.
  - 포트가 메모리나 MCU(제품 동작을 제어하는 핵심 칩)와 직접 연결되어 있다면, 내부 구조를 들여다볼 수 있는 경로가 됩니다.

- **문제가 될 수 있는 하드웨어 구조**
  - 개발용 포트를 양산 PCB에 그대로 남겨두면, 제품을 뜯은 사람이 내부 칩과 통신할 가능성이 생깁니다.
  - 참고 자료에서도 **외부에 인터페이스가 존재하거나, 해체 후 내부 PCB의 메모리 및 MCU에 접근 가능한 포트가 있는 제품**은 물리적 보안 점검 대상이라고 설명합니다.
  - 즉, 포트 자체가 있느냐보다 **그 포트가 실제로 내부 핵심 부품에 접근할 수 있는 통로인지**가 핵심입니다.

- **설계상 개선할 수 있는 부분**
  - 양산 제품에서는 개발용 포트를 **제거하거나 비활성화**하는 방향을 우선 검토하는 것이 좋습니다.
  - 꼭 필요하다면, **일반 사용자가 쉽게 접근할 수 없도록 내부에 숨기거나 접근을 제한**해야 합니다.
  - 제품을 분해해야만 닿는 위치에 있더라도, 그 포트가 핵심 부품과 연결되어 있다면 별도 보호가 필요합니다.

### 보안 점검

- **왜 위험할 수 있는지**
  - 개발용 포트는 원래 내부 상태를 확인하거나 통신하기 위한 통로라서, 남아 있으면 공격자가 이를 이용해 **내부 프로그램이나 저장된 정보를 읽어갈 가능성**이 생깁니다.
  - 스마트 도어락은 출입 제어와 인증정보가 중요한 제품이므로, 이런 통로가 노출되면 단순 편의 기능이 아니라 **무단 조작과 직접 연결된 위험**이 됩니다.

- **어떤 공격이나 정보 유출로 이어질 수 있는지**
  - 분해한 뒤 포트를 통해 내부 상태를 확인하거나, 펌웨어나 설정값을 노출시킬 수 있습니다.
  - 인증과 접근통제 측면에서도, 관리자가 아니어도 내부 접근 경로를 통해 **등록 정보나 제어 관련 정보를 악용**할 가능성이 있습니다.
  - 참고 자료의 접근통제 원칙상, **중요 기능은 인가된 관리자만 접근**해야 하므로, 개발용 포트가 그 제한을 우회하는 수단이 되면 문제가 됩니다.

- **필요한 보호 방법**
  - 양산 단계에서는 개발용 포트를 남겨둘 필요가 있는지 먼저 판단해야 합니다.
  - 남겨야 한다면, **일반 사용자에게 노출되지 않도록 물리적으로 통제**하고, 중요 기능 접근에 사용되지 않도록 해야 합니다.
  - 참고 자료에 따르면, 중요한 기능은 **사용자 인증 후 인가된 관리자/사용자로 제한**해야 하므로, 포트 접근만으로 우회되지 않게 해야 합니다.

### 통합 개선안

1. **양산 PCB에서 개발용 포트가 꼭 필요한지 먼저 확인**
   - 필요 없으면 제거 또는 비활성화하는 것이 가장 안전합니다.

2. **포트가 내부 메모리·MCU 접근 통로인지 점검**
   - 내부 프로그램, 설정값, 등록정보로 이어질 수 있으면 보안 위험이 커집니다.

3. **중요 기능은 사용자 인증과 권한 통제로만 접근**
   - 참고 자료의 접근통제 원칙처럼, 일반 사용자가 관리자 기능에 접근하지 못하게 해야 합니다.

4. **물리적 접근 자체를 줄이는 구조로 변경**
   - 제품 구매자만 접근 가능한 공간에 설치하고, 무선통신이나 외부 포트를 통해 쉽게 설정되지 않도록 하는 방향이 좋습니다.

5. **분해 시 드러나는 핵심 부품 보호 검토**
   - 기기를 열었을 때 MCU나 메모리에 바로 접근되지 않도록 설계상 배치를 다시 확인해야 합니다.

정리하면, **개발용 포트를 그대로 남겨두는 것은 권장되지 않습니다.**  
참고 자료만으로는 포트의 정확한 용도와 연결 구조를 알 수 없지만, **내부 PCB의 메모리·MCU에 접근 가능한 포트라면 물리적 보안상 우선 점검 대상**입니다.

### 참고 자료

- KISA_홈가전IoT보안가이드.pdf, p.23
- KISA_홈가전IoT보안가이드.pdf, p.62


질문: 홈캠 안에 영상이나 비밀번호 같은 정보를 저장하려는데, 하드웨어는 어떻게 구성하고 정보는 어떻게 보호해야 해?



### 종합 점검 결과

홈캠은 **영상정보를 저장·전송하는 제품**이기 때문에, 저장장치가 어디에 있고 기판에서 얼마나 쉽게 접근되는지가 매우 중요합니다.  
또한 **비밀번호, 인증정보, 영상 같은 중요정보는 일반 저장공간에 두는 순간 물리적 분해나 디버그 포트 노출을 통해 읽힐 가능성**이 생기므로, 하드웨어 구조와 정보 보호를 같이 설계해야 합니다.

---

### 하드웨어 설계 점검

#### 1) 현재 구조에서 우선 확인할 부분
홈캠은 참고 자료상 **촬영 제품**에 해당합니다. 따라서 확인해야 할 핵심은 다음입니다.

- 영상이 **기기 내부에 저장되는지**, 아니면 네트워크로만 전송되는지
- 저장된다면 **어떤 저장장치에 저장되는지**
- 분해했을 때 저장장치, MCU(제품 동작을 제어하는 핵심 칩), 통신선에 **쉽게 접근 가능한지**
- 기판에 **UART, JTAG 같은 개발·점검용 포트**가 남아 있는지
- 영상 처리나 인증정보 처리에 필요한 부품이 **별도 보안 하드웨어 없이 일반 저장공간에만 의존하는지**

#### 2) 하드웨어 구조상 문제가 될 수 있는 부분
참고 자료만으로 홈캠의 실제 회로를 특정할 수는 없지만, 일반적으로 다음 구조는 주의가 필요합니다.

- **영상이나 비밀번호가 일반 메모리/저장공간에 저장되는 구조**
  - 분해 후 저장장치를 직접 읽을 가능성이 생깁니다.
- **개발용 포트가 양산 제품에 남아 있는 구조**
  - 내부 프로그램이나 저장정보를 확인하는 통로가 될 수 있습니다.
- **중요 통신선이 외부에서 쉽게 식별되는 구조**
  - MCU와 저장장치 사이의 통신이 노출되면 분석이 쉬워집니다.
- **테스트 포인트가 쉽게 노출된 구조**
  - 점검용 접점이 공격자의 진입점이 될 수 있습니다.

#### 3) 설계상 개선 방향
- 양산 PCB에서는 **개발용 포트 노출을 최소화**하거나 비활성화하는 방향이 좋습니다.
- 영상 저장이 필요하다면, **일반 저장공간과 중요 정보 저장 영역을 분리**하는 구성이 바람직합니다.
- 비밀번호나 인증정보처럼 중요한 정보는 **외부에서 쉽게 읽기 어려운 별도 하드웨어 보호 수단**을 고려할 수 있습니다.
- 분해 시 주요 부품에 쉽게 닿지 않도록, **핵심 부품과 테스트 포인트의 물리적 노출을 줄이는 배치**가 필요합니다.

---

### 보안 점검

#### 1) 무엇이 문제인지
홈캠은 영상정보와 인증정보를 다루므로, 저장 방식이 약하면 **개인영상 유출**이나 **계정·비밀번호 도용**으로 이어질 수 있습니다.  
참고 자료에서도 촬영 제품의 보안성으로 **개인영상의 기밀성**, **인가된 사용자의 저장 영상 접근**, **인증정보의 기밀성 및 무결성**을 요구합니다.

#### 2) 어떤 하드웨어 설계 때문에 위험해지는지
- 영상이나 비밀번호를 **일반 저장장치에 그대로 저장**하면 물리적으로 꺼내 읽기 쉬워집니다.
- **UART/JTAG 같은 포트가 남아 있으면** 내부 상태 확인이나 프로그램 추출의 통로가 될 수 있습니다.
- **MCU와 저장장치 사이 연결이 노출되면** 역공학이나 데이터 추적이 쉬워집니다.
- **중요 데이터와 일반 데이터가 분리되지 않으면** 한 번 접근당했을 때 피해 범위가 커집니다.

#### 3) 실제로 어떤 일이 발생할 수 있는지
- 저장된 **개인영상이 외부로 유출**될 수 있습니다.
- **비밀번호나 인증정보가 도용**되어 원격 접속이나 기기 제어가 가능해질 수 있습니다.
- 내부 프로그램이 추출되면 **펌웨어 변조**나 기능 분석이 쉬워질 수 있습니다.
- 분해 후 저장소를 직접 읽어 **사용자 정보가 복구**될 수 있습니다.

#### 4) 필요한 보호 방법
- 중요정보는 **암호화**하여 저장하는 것이 필요합니다.
- 암호화에 쓰는 **암호키는 일반 저장공간에 두지 않는 것**이 중요합니다.
- 참고 자료에 따르면 난수 생성 시에는 **국내외 표준에 따른 안전한 난수 발생기**를 사용해야 합니다.  
  즉, 키를 만들 때 예측 가능한 난수를 쓰면 보안성이 약해질 수 있습니다.
- 업데이트가 있는 제품이라면, **안전한 업데이트 절차**도 함께 고려해야 합니다.
- 디버그 포트와 테스트 포인트는 **양산 제품에서 접근이 어렵게** 해야 합니다.

---

### 통합 개선안

가장 중요한 순서대로 정리하면 다음과 같습니다.

1. **영상·비밀번호 같은 중요정보를 일반 저장공간에 그대로 두지 않기**
   - 최소한 암호화 저장을 검토해야 합니다.
   - 암호키는 별도로 보호해야 합니다.

2. **양산 PCB에서 개발용 포트 노출 최소화**
   - UART, JTAG 같은 점검용 통로가 남아 있으면 공격 표면이 커집니다.
   - 필요 시 비활성화하거나 접근을 어렵게 해야 합니다.

3. **중요 저장공간과 일반 기능 영역 분리**
   - 영상 저장, 인증정보, 운영 데이터가 섞이면 침해 시 피해가 커집니다.
   - 저장 구조를 분리하면 유출 범위를 줄일 수 있습니다.

4. **안전한 난수 생성 방식 사용**
   - 암호키 생성에 쓰는 난수가 약하면 전체 암호화가 무력화될 수 있습니다.

5. **분해·물리 공격을 고려한 배치**
   - 저장장치, 핵심 통신선, 테스트 포인트가 쉽게 드러나지 않도록 구성하는 것이 좋습니다.

6. **업데이트 무결성 검토**
   - 펌웨어가 바뀔 수 있는 제품이라면, 무단 변조를 막는 절차가 필요합니다.

현재 참고 자료만으로는 홈캠의 실제 PCB 구성, 저장장치 종류, 포트 유무를 정확히 판단하기 어렵습니다.  
하지만 자료에 근거하면, **“개인영상과 인증정보를 다루는 촬영 제품”이므로 저장 구조와 디버그 포트 관리가 핵심**입니다.

---

### 참고 자료

- KISA_홈가전IoT보안가이드.pdf, p.18
- KISA_홈가전IoT보안가이드.pdf, p.78


질문: 스마트 플러그 기판을 설계할 때 외부에서 쉽게 건드릴 수 있는 포트나 통신선은 어떻게 처리하는 게 좋아?



...

### 종합 점검 결과

양산용 IoT 제품에서는 **개발용 기판에 남아 있는 디버그·점검용 연결 통로를 줄이고**, 동시에 **인증, 암호화, 데이터 보호, 안전한 업데이트**가 실제 하드웨어 구조와 맞물려 동작하는지 확인하는 것이 핵심입니다.  
특히 개발 단계에서 편의를 위해 남겨둔 포트나 테스트 포인트가 양산 제품에 그대로 남으면, 기기를 분해했을 때 내부 프로그램이나 저장된 정보에 접근할 가능성이 생깁니다.

---

### 하드웨어 설계 점검

#### 1) 개발용 기판에서 양산용 기판으로 바꿔야 할 부분
참고 자료에는 홈·가전 IoT 제품의 공통 보안항목으로 **물리적 보안**과 **소프트웨어 보안**이 제시되어 있고, 물리적 보안 항목으로는 **“물리적 인터페이스 차단”**이 포함되어 있습니다.  
즉, 개발 단계에서 사용하던 외부 연결 통로를 양산 제품에서 그대로 두는 것은 적절하지 않을 수 있습니다.

확인해야 할 부분은 다음과 같습니다.

- **UART, JTAG 같은 개발·점검용 포트가 그대로 남아 있는지**
  - 이런 포트는 기기 내부 상태를 확인하거나 프로그램을 점검하는 데 쓰일 수 있습니다.
  - 양산 제품에서는 꼭 필요한 경우가 아니면 제거하거나 접근을 어렵게 해야 합니다.

- **테스트 포인트가 외부에서 쉽게 보이거나 닿는지**
  - 기기를 열었을 때 신호선이나 점검용 접점이 바로 드러나면 분해 후 분석이 쉬워집니다.
  - 참고 자료의 “물리적 인터페이스 차단” 관점에서 노출을 줄이는 방향이 필요합니다.

- **MCU와 저장장치, 보안 관련 부품의 연결 구조**
  - MCU(제품 동작을 제어하는 핵심 칩)와 메모리, 통신 부품이 어떻게 연결되는지 확인해야 합니다.
  - 중요한 정보가 일반 저장공간에 몰려 있으면, 분해 후 접근 위험이 커집니다.

- **분해 시 주요 부품에 쉽게 접근되는지**
  - 제품 구조상 나사를 풀거나 커버를 열면 바로 핵심 부품이 보이는 구조라면 물리적 공격에 더 취약할 수 있습니다.

#### 2) 하드웨어 설계상 개선 방향
참고 자료에 근거해 볼 때, 다음과 같은 방향이 적절합니다.

- **개발용 포트는 양산 PCB에서 제거하거나 비활성화**
- **필요 없는 내부 인터페이스는 차단**
- **테스트 포인트 노출 최소화**
- **중요 데이터와 일반 저장공간을 분리하는 구조 검토**
- **인증·암호화 관련 기능이 하드웨어 구조와 맞게 동작하도록 설계**

---

### 보안 점검

참고 자료에서 유형별 보안항목으로 **인증, 암호화, 데이터 보호, 플랫폼 보안**이 제시되어 있습니다.  
또한 요구사항으로는 **안전한 암호 알고리즘 사용, 안전한 암호키 관리, 안전한 통신채널, 저장 및 전송 데이터 보호, 설정값 및 실행코드 무결성 검증, 안전한 업데이트, 개인정보 보호, 인증 및 접근통제, IoT 제품 간 상호인증** 등이 포함됩니다.

#### 1) 개발용 포트가 남아 있을 때의 보안 위험
- **무엇이 문제인지**
  - 개발·점검용 포트가 양산 제품에 남으면 내부에 접근할 통로가 생깁니다.

- **어떤 하드웨어 설계 때문에 발생하는지**
  - UART, JTAG 같은 포트가 외부에서 접근 가능하거나, 테스트 포인트가 노출된 경우입니다.

- **왜 위험한지**
  - 공격자가 이 경로를 이용해 내부 상태를 확인하거나, 저장된 데이터나 프로그램을 분석할 가능성이 생깁니다.

- **실제로 어떤 일이 발생할 수 있는지**
  - 내부 프로그램 추출, 설정값 확인, 무단 조작 시도, 펌웨어 분석 가능성이 커집니다.

- **개선 방법**
  - 양산 제품에서는 해당 포트를 제거하거나 접근을 제한하고, 물리적 인터페이스를 차단해야 합니다.

#### 2) 암호화와 암호키 관리
- **무엇이 문제인지**
  - 중요한 데이터가 저장되거나 전송될 때 보호가 없으면 유출될 수 있습니다.

- **어떤 하드웨어 설계 때문에 발생하는지**
  - 저장장치와 MCU 사이의 구조가 분해 후 쉽게 분석되거나, 중요한 정보가 일반 저장공간에 그대로 남는 경우입니다.

- **왜 위험한지**
  - 개인정보, 인증정보, 제어정보가 읽히면 제품이 무단으로 사용될 수 있습니다.

- **실제로 어떤 일이 발생할 수 있는지**
  - 계정 정보 탈취, 설정 변경, 원격 제어, 데이터 위변조가 가능합니다.

- **개선 방법**
  - 안전한 암호 알고리즘을 사용하고, 암호키는 안전하게 관리해야 합니다.
  - 참고 자료만으로는 별도 하드웨어 보안 모듈의 필요 여부를 단정할 수 없지만, 중요한 암호키는 일반 저장공간에 두지 않는 방향이 바람직합니다.

#### 3) 설정값과 실행코드 보호
- **무엇이 문제인지**
  - 펌웨어나 설정값이 변조되면 제품 동작 자체가 바뀔 수 있습니다.

- **어떤 하드웨어 설계 때문에 발생하는지**
  - 저장장치 접근이 쉬운 구조, 디버그 포트 노출, 분해가 쉬운 구조입니다.

- **왜 위험한지**
  - 공격자가 내부 프로그램을 바꾸거나 악성 펌웨어를 넣을 수 있습니다.

- **실제로 어떤 일이 발생할 수 있는지**
  - 무단 제어, 기능 오작동, 업데이트 악용 가능성이 있습니다.

- **개선 방법**
  - 설정값 및 실행코드 무결성 검증을 적용하고, 안전한 업데이트가 가능해야 합니다.

---

### 통합 개선안

가장 우선적으로는 **양산 PCB에서 개발용 포트와 테스트 포인트 노출을 줄이는 것**이 필요합니다.  
그 다음으로는 **중요 데이터의 저장·전송 보호**, **암호키 관리**, **설정값 및 실행코드 무결성 검증**, **안전한 업데이트**가 실제 하드웨어 구조와 함께 동작하는지 확인해야 합니다.

권장 순서는 다음과 같습니다.

1. **UART/JTAG 등 개발용 포트 제거 또는 비활성화**
2. **물리적 인터페이스 차단 및 테스트 포인트 노출 최소화**
3. **중요 정보의 저장 위치와 일반 데이터의 분리 검토**
4. **안전한 암호 알고리즘 및 암호키 관리 적용**
5. **저장·전송 데이터 보호 적용**
6. **설정값 및 실행코드 무결성 검증**
7. **안전한 업데이트 체계 확인**
8. **필요 시 인증 및 접근통제, IoT 제품 간 상호인증 검토**

현재 참고 자료만으로는 MCU 종류, 메모리 구성, 보안 모듈의 유무, 특정 포트의 실제 배치까지는 확인할 수 없습니다. 따라서 **구체적인 회로 수치나 부품 사양은 판단하기 어렵고**, 실제 양산 전에는 기판 실물 기준으로 포트 노출, 분해 용이성, 저장장치 접근성을 함께 점검하는 것이 좋습니다.

---

### 참고 자료

- KISA_홈가전IoT보안가이드.pdf, p.19
- KISA_홈가전IoT보안가이드.pdf, p.20


질문: IoT 기판에서 MCU와 보안용 칩을 연결하려면 어떤 통신 방식을 사용할 수 있어?



### 종합 점검 결과

현재 참고 자료상으로는, IoT 기판에서 **MCU(제품의 동작을 제어하는 핵심 칩)** 와 **보안용 칩(하드웨어 보안 모듈, HSM/SE 등)** 을 **I2C, UART, SPI 같은 직렬 통신 방식**으로 연결하는 구성이 일반적입니다.  
중요한 점은 단순히 “연결한다”가 아니라, **내부 통신 구간을 안전하게 보호해 키나 인증정보가 노출되지 않도록 설계하는 것**입니다.

---

### 하드웨어 설계 점검

#### 1) 현재 설계에서 확인해야 할 부분
참고 자료에 따르면, 하드웨어 보안 기술은 보통 **MCU 주변의 별도 하드웨어 모듈**로 보드에 함께 올라가고, MCU와의 통신은 **Serial 방식(I2C, UART, SPI 등)** 으로 이루어집니다.  
즉, 설계할 때는 다음을 확인해야 합니다.

- 보안용 칩이 MCU와 **어떤 직렬 통신선**으로 연결되는지
- 그 통신선이 **보드 위에서 쉽게 식별되거나 접근 가능한지**
- 민감한 데이터가 **MCU 메인 메모리나 일반 저장공간을 거치지 않고** 보안용 칩에서 처리되는지

#### 2) 문제가 될 수 있는 하드웨어 구조
보안용 칩을 MCU와 연결할 때, 통신선이 단순하게 노출되면 외부에서 분해 후 분석하기 쉬워집니다.  
특히 참고 자료에는 **MCU와 하드웨어 보안 모듈 간 내부 통신이 안전한 채널로 구성되어야 한다**고 되어 있으므로, 통신 방식 자체보다 **그 통신이 보안적으로 보호되는지**가 핵심입니다.

#### 3) 설계상 개선할 수 있는 부분
- 보안용 칩은 **MCU와 같은 보드상에 두되**, 내부 민감정보가 일반 회로로 퍼지지 않게 구성
- I2C/UART/SPI 같은 통신은 사용하되, **내부 보안 채널**을 함께 고려
- 민감한 데이터는 가능한 한 **물리적으로 분리된 보안용 칩에 저장**
- 제품 분해 시 통신선이 쉽게 드러나지 않도록 배치와 노출을 줄이는 방향 검토

---

### 보안 점검

#### 1) 무엇이 문제인지
MCU와 보안용 칩이 직렬 통신으로 연결되면, 그 구간을 통해 **키, 인증정보, 개인정보 같은 민감한 데이터**가 오갈 수 있습니다.  
이 통신이 보호되지 않으면 공격자가 분해 후 신호를 관찰하거나 통신을 따라가며 정보를 얻을 가능성이 생깁니다.

#### 2) 어떤 하드웨어 설계 때문에 문제가 발생하는지
- 보안용 칩과 MCU가 **보드 내부에서 직렬 통신으로 직접 연결**됨
- 내부 통신 구간이 **암호화·인증 없이 평문 수준으로 동작**할 수 있음
- 물리적으로 분해했을 때 **연결 구조를 추적하기 쉬울 수 있음**

#### 3) 왜 보안상 위험한지
참고 자료에서는 내부 상호 통신(예: ISO 7816 APDU 방식)에서도 **단방향 및 양방향 인증을 통해 안전한 내부 보안 채널을 구성**할 수 있다고 설명합니다.  
즉, 단순 연결만으로는 충분하지 않고, **기밀성과 무결성**이 보장되어야 합니다.

#### 4) 실제로 어떤 일이 발생할 수 있는지
- 보안용 칩에서 저장된 키가 호출되는 과정이 노출될 수 있음
- 인증정보가 중간에서 관찰되거나 변조될 수 있음
- 공격자가 내부 통신을 흉내 내거나 가로채서 제품 기능을 악용할 수 있음

#### 5) 하드웨어 또는 보안 설정을 어떻게 개선하면 좋은지
- MCU와 보안용 칩 사이에 **상호인증**을 적용
- 내부 통신 구간에 **암호화와 무결성 보호**를 적용
- 보안용 칩에 저장된 키는 **안전한 내부 채널을 통해서만 호출·사용**되도록 구성
- 통신선과 관련 신호를 **물리적으로 쉽게 접근하기 어렵게** 배치

---

### 통합 개선안

1. **보안용 칩과 MCU는 I2C, UART, SPI 같은 직렬 통신으로 연결할 수 있습니다.**  
   다만 이때 단순 연결이 아니라, 내부 통신 자체를 보호하는 설계가 필요합니다.

2. **민감정보는 MCU 일반 영역이 아니라 보안용 칩에 분리 저장**하는 방향이 좋습니다.  
   이렇게 하면 키, 인증정보, 개인정보가 메인 MCU 쪽으로 넓게 퍼지는 것을 줄일 수 있습니다.

3. **MCU-보안용 칩 사이에는 인증과 암호화가 포함된 내부 보안 채널**을 구성하는 것이 중요합니다.  
   참고 자료에 따르면 단방향/양방향 인증과 기밀성·무결성 보호가 핵심입니다.

4. **제품 분해 후 통신선이 쉽게 드러나지 않도록 보드 배치와 노출을 최소화**하세요.  
   하드웨어 구조가 단순할수록 물리적 분석이 쉬워질 수 있으므로, 개발 편의성과 보안 사이 균형이 필요합니다.

---

### 참고 자료

- KISA_홈가전IoT보안가이드.pdf, p.143
- KISA_홈가전IoT보안가이드.pdf, p.83


질문: 기판 안에서 중요한 통신선을 어떻게 배치하는 게 좋아?



### 종합 점검 결과

기판 안의 중요한 통신선은 **외부에서 쉽게 찾거나 건드리기 어렵게** 배치하는 것이 좋습니다.  
특히 분해했을 때 바로 보이는 통신선은 도청이나 변조의 출발점이 될 수 있으므로, **접근성 최소화와 안전한 통신채널**을 함께 고려해야 합니다.

### 하드웨어 설계 점검

- 질문의 핵심은 **기판 내부의 중요한 통신선 배치**입니다.
- 참고 자료에서는 홈·가전 IoT 제품에서 전송되는 중요 데이터를 제3자가 도청·위변조하지 못하도록 **안전한 암호화 통신채널**을 제공해야 한다고 설명합니다.
- 따라서 하드웨어 관점에서는 다음을 우선 확인해야 합니다.
  - 중요한 통신선이 **기판 표면에서 쉽게 식별되는지**
  - 분해했을 때 **테스트 포인트나 배선이 바로 노출되는지**
  - MCU(제품 동작을 제어하는 핵심 칩)와 통신 대상 부품 사이의 연결이 **외부에서 쉽게 접근 가능한지**
  - 개발·점검용 포트와 함께 **내부 통신선이 지나치게 가까이 배치되어 있는지**

- 설계상으로는 중요한 통신선을
  - 외부에서 바로 찍기 쉬운 위치에 두기보다
  - **분해 후에도 바로 손대기 어렵게** 내부 배선 구조를 고려하고
  - 테스트용 접점과 분리해 배치하는 방향이 바람직합니다.

- 다만 참고 자료만으로는 **기판 내 구체적인 배선 위치, 층수, 간격, 패턴 폭**까지는 확인할 수 없습니다.  
  따라서 현재 자료만으로는 “어느 위치에, 어떤 형태로” 배치해야 하는지의 상세 회로 수준 판단은 어렵습니다.

### 보안 점검

- 중요한 통신선이 쉽게 드러나면 공격자가 이를 이용해 **도청**하거나 **신호를 변조**할 수 있습니다.
- 참고 자료에서도 중요 데이터는 제3자가 가로채거나 바꾸지 못하도록 **암호화 통신채널**이 필요하다고 설명합니다.
- 즉, 하드웨어 배치가 나쁘면 암호화가 있더라도 공격자가
  - 통신 흐름을 관찰하거나
  - 내부 신호를 분석하거나
  - 테스트 포인트를 통해 신호를 주입하는 식의 공격을 시도할 수 있습니다.
- 이 경우 발생할 수 있는 피해는 다음과 같습니다.
  - 중요 정보 유출
  - 기기 제어 신호 위변조
  - 인증 정보 노출 가능성 확대
- 따라서 중요한 통신선은 **외부 접근성 최소화**와 함께, 실제 데이터 자체는 **암호화된 통신채널**로 보호하는 것이 중요합니다.

### 통합 개선안

1. **중요한 통신선을 외부에서 쉽게 보이지 않게 배치**
   - 분해 시 바로 식별되는 경로를 줄입니다.
2. **테스트 포인트와 주요 신호선 분리**
   - 점검용 접점이 통신선과 가까우면 공격에 악용될 수 있습니다.
3. **중요 데이터는 암호화 통신으로 전송**
   - 하드웨어 배치만이 아니라 데이터 자체 보호가 필요합니다.
4. **개발·점검용 포트와 양산용 구성을 구분**
   - 양산 제품에는 불필요한 접근 통로가 남지 않도록 점검합니다.
5. **분해 후 접근성을 기준으로 재검토**
   - 제품을 열었을 때 어떤 신호가 바로 보이는지 실제 기준으로 확인하는 것이 좋습니다.

### 참고 자료

- KISA_홈가전IoT보안가이드.pdf, p.80

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합